# Healthy Economies, Healthy Communities
## Full Analysis Notebook
### Mastercard IGS × AUC 2026 Data Science Challenge

**Author:** Saurav Bhattarai (Newton), Jackson State University  
**Date:** April 2026  

This notebook reproduces the full analytical story end-to-end using the **actual pipeline outputs** in `data_processed/`. No values are hardcoded — every number is read from the built parquet files or derived on the fly from IGS data.

**Run order prerequisite:**
```
python 01_ingest/run_all_ingest.py
python 02_build/build_master_tract.py
python 02_build/build_igs_trends.py
python 02_build/build_igs_national.py
python 02_build/build_delta_profile.py
python 03_analysis/run_all_analysis.py
```

## 0. Setup

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import pearsonr

from config import (
    MASTER_TRACT, IGS_TRENDS_PARQUET, PROCESSED,
    DELTA_COUNTY_FIPS, DELTA_COUNTY_NAMES,
    IGS_VULN_THRESHOLD, IGS_SUB_TO_PILLAR,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Setup complete.')

## 1. Load Data

In [ ]:
# ── Core datasets ──────────────────────────────────────────────────────────
df     = pd.read_parquet(MASTER_TRACT)
trends = pd.read_parquet(IGS_TRENDS_PARQUET)

# ── SHAP outputs (generated by ml_discovery.py) ────────────────────────────
shap_weights_path = PROCESSED / 'shap_derived_weights.json'
shap_summary_path = PROCESSED / 'shap_feature_summary.parquet'
shap_dims_path    = PROCESSED / 'shap_dimensions.parquet'
regression_path   = PROCESSED / 'igs_regression_coefficients.parquet'
typology_path     = PROCESSED / 'community_typology.parquet'
benchmarks_path   = PROCESSED / 'turnaround_benchmarks.parquet'

shap_weights  = json.loads(shap_weights_path.read_text()) if shap_weights_path.exists() else {}
shap_summary  = pd.read_parquet(shap_summary_path) if shap_summary_path.exists() else None
shap_dims     = pd.read_parquet(shap_dims_path)    if shap_dims_path.exists()    else None
reg_coefs     = pd.read_parquet(regression_path)   if regression_path.exists()   else None
typology      = pd.read_parquet(typology_path)     if typology_path.exists()     else None
benchmarks    = pd.read_parquet(benchmarks_path)   if benchmarks_path.exists()   else None

# ── Delta subset ───────────────────────────────────────────────────────────
if 'county_fips5' in df.columns:
    delta_df = df[df['county_fips5'].isin(DELTA_COUNTY_FIPS)].copy()
else:
    delta_df = pd.DataFrame()

print(f'Master dataset:  {len(df):,} tracts × {df.shape[1]} columns')
print(f'IGS trends:      {len(trends):,} rows | years: {sorted(trends["year"].unique())}')
print(f'Delta tracts:    {len(delta_df):,}')
print(f'SHAP dimensions loaded: {list(shap_weights.keys()) or "(run ml_discovery.py)"}')
print(f'Regression coefs: {"loaded" if reg_coefs is not None else "(run igs_regression_analysis.py)"}')
print(f'Typology:         {"loaded" if typology is not None else "(run community_typology.py)"}')

## 2. Triple-Vulnerability Analysis

We define three independent vulnerability dimensions from the actual data:
- **Economic** — IGS score below the threshold (`IGS_VULN_THRESHOLD` from config)
- **Climate** — FEMA NRI composite risk score in the top quartile nationally
- **Health** — Located in a federally designated Primary Care or Mental Health shortage area (HPSA score > 0)

In [ ]:
# Economic vulnerability — defined by the project's own threshold
df['econ_vuln'] = df['igs_score'] < IGS_VULN_THRESHOLD

# Climate vulnerability — top quartile FEMA NRI risk (threshold is data-driven)
if 'RISK_SCORE' in df.columns:
    risk_q75 = df['RISK_SCORE'].quantile(0.75)
    df['climate_vuln'] = df['RISK_SCORE'] > risk_q75
    print(f'Climate vulnerability threshold (75th pct of RISK_SCORE): {risk_q75:.2f}')
else:
    df['climate_vuln'] = False
    print('WARNING: RISK_SCORE not in master_tract — re-run build_master_tract.py')

# Health shortage area — any HPSA designation (PC or MH)
hpsa_cols = [c for c in ['pc_hpsa_score_max', 'mh_hpsa_score_max'] if c in df.columns]
if hpsa_cols:
    df['health_vuln'] = (df[hpsa_cols] > 0).any(axis=1)
    print(f'Health shortage using: {hpsa_cols}')
else:
    df['health_vuln'] = False
    print('WARNING: HPSA columns not found — re-run build_master_tract.py')

df['triple_vuln'] = df['econ_vuln'] & df['climate_vuln'] & df['health_vuln']

# Propagate flags to delta_df
if not delta_df.empty:
    for col in ['econ_vuln', 'climate_vuln', 'health_vuln', 'triple_vuln']:
        delta_df[col] = delta_df.index.map(df[col])

n_total = len(df)
print()
print('=== NATIONAL TRIPLE-VULNERABILITY SUMMARY ===')
for label, col in [
    (f'Economically vulnerable (IGS < {IGS_VULN_THRESHOLD})', 'econ_vuln'),
    ('Climate exposed (top 25% FEMA NRI risk)',                'climate_vuln'),
    ('Health shortage area (HPSA PC or MH)',                   'health_vuln'),
    ('Triple-vulnerable (all three)',                          'triple_vuln'),
]:
    n = int(df[col].sum())
    print(f'  {label:<48} {n:>8,}  ({n/n_total*100:.1f}%)')

pop_col = next((c for c in ['POPULATION', 'E_TOTPOP'] if c in df.columns), None)
if pop_col:
    pop_triple = df[df['triple_vuln']][pop_col].sum()
    print(f'\n  Population in triple-vulnerable tracts: {pop_triple/1e6:.1f}M')

## 3. Vulnerability Overlap Matrix

In [ ]:
e, c, h = df['econ_vuln'], df['climate_vuln'], df['health_vuln']

print('=== VULNERABILITY OVERLAP MATRIX ===')
for label, mask in [
    ('Only Economic',       e  & ~c & ~h),
    ('Only Climate',        ~e &  c & ~h),
    ('Only Health',         ~e & ~c &  h),
    ('Economic + Climate',   e &  c & ~h),
    ('Economic + Health',    e & ~c &  h),
    ('Climate + Health',    ~e &  c &  h),
    ('Triple (all three)',   e &  c &  h),
]:
    n = int(mask.sum())
    print(f'  {label:<25}  {n:>8,}  ({n/n_total*100:.1f}%)')

# State rankings by triple-vulnerability rate
df['STATE'] = df['GEOID'].str[:2]
state_summary = df.groupby('STATE').agg(
    total_tracts   =('GEOID', 'count'),
    triple_vuln_n  =('triple_vuln', 'sum'),
    mean_igs       =('igs_score', 'mean'),
).reset_index()
state_summary['triple_pct'] = state_summary['triple_vuln_n'] / state_summary['total_tracts'] * 100
state_summary = state_summary.sort_values('triple_pct', ascending=False)

print('\nTop 10 States by Triple-Vulnerability Rate:')
print(state_summary[['STATE','total_tracts','triple_vuln_n','triple_pct','mean_igs']].head(10).to_string(index=False))

## 4. IGS Score Distribution: National vs. MS Delta

In [ ]:
nat_igs   = df['igs_score'].mean()
delta_igs = delta_df['igs_score'].mean() if not delta_df.empty else float('nan')

print(f'National mean IGS:  {nat_igs:.1f}')
print(f'MS Delta mean IGS:  {delta_igs:.1f}')
print(f'Gap:                {delta_igs - nat_igs:.1f} points below national')
print(f'Tracts below {IGS_VULN_THRESHOLD}:   {int(df["econ_vuln"].sum()):,} of {n_total:,} ({df["econ_vuln"].mean()*100:.1f}%)')

if not delta_df.empty:
    print('\nDelta county IGS scores:')
    county_igs = (
        delta_df.groupby('county_fips5')['igs_score'].mean()
        .reset_index()
        .assign(county_name=lambda x: x['county_fips5'].map(DELTA_COUNTY_NAMES))
        .sort_values('igs_score')
    )
    for _, row in county_igs.iterrows():
        below = '  ← below threshold' if row['igs_score'] < IGS_VULN_THRESHOLD else ''
        print(f"  {row['county_name']:<15}  {row['igs_score']:.1f}{below}")

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df['igs_score'].dropna(), nbinsx=60, name='All US Tracts',
    marker_color='steelblue', opacity=0.5, histnorm='probability density',
))
if not delta_df.empty:
    fig.add_trace(go.Histogram(
        x=delta_df['igs_score'].dropna(), nbinsx=20, name='MS Delta Tracts',
        marker_color='#d73027', opacity=0.8, histnorm='probability density',
    ))
fig.add_vline(x=nat_igs,           line_dash='dash', line_color='steelblue',
              annotation_text=f'National mean ({nat_igs:.1f})')
fig.add_vline(x=delta_igs,         line_dash='dash', line_color='#d73027',
              annotation_text=f'Delta mean ({delta_igs:.1f})')
fig.add_vline(x=IGS_VULN_THRESHOLD, line_dash='dot',  line_color='orange',
              annotation_text=f'Vulnerability threshold ({IGS_VULN_THRESHOLD})')
fig.update_layout(
    barmode='overlay',
    title='Distribution of IGS Scores: All US Tracts vs. MS Delta',
    xaxis_title='IGS Score (0–100)', yaxis_title='Density', height=420,
)
fig.show()

## 5. IGS Trend Analysis (2017–2025)

In [ ]:
national_trend = trends.groupby('year')['igs_score'].mean().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=national_trend['year'], y=national_trend['igs_score'],
    name='National Average', line=dict(color='steelblue', width=3), mode='lines+markers',
))

if 'county_fips5' in trends.columns:
    cf = trends['county_fips5'].astype(str).str.strip().str.zfill(5)

    ms_trend = trends[cf.str.startswith('28', na=False)].groupby('year')['igs_score'].mean().reset_index()
    fig.add_trace(go.Scatter(
        x=ms_trend['year'], y=ms_trend['igs_score'],
        name='Mississippi Average', line=dict(color='#ca8a04', width=2), mode='lines+markers',
    ))

    delta_trend = trends[cf.isin(DELTA_COUNTY_FIPS)].groupby('year')['igs_score'].mean().reset_index()
    fig.add_trace(go.Scatter(
        x=delta_trend['year'], y=delta_trend['igs_score'],
        name='MS Delta (9 counties)', line=dict(color='#d73027', width=3), mode='lines+markers',
    ))

    merged = national_trend.merge(delta_trend, on='year', suffixes=('_nat', '_delta'))
    merged['gap'] = merged['igs_score_nat'] - merged['igs_score_delta']
    print('IGS gap (National − Delta) by year:')
    for _, row in merged.iterrows():
        print(f"  {int(row['year'])}: National={row['igs_score_nat']:.1f}  "
              f"Delta={row['igs_score_delta']:.1f}  Gap={row['gap']:.1f}")

fig.add_hline(
    y=IGS_VULN_THRESHOLD, line_dash='dash', line_color='orange',
    annotation_text=f'Vulnerability threshold ({IGS_VULN_THRESHOLD})',
)
fig.update_layout(
    title='IGS Score Trend 2017–2025: National vs. Mississippi vs. Delta',
    xaxis_title='Year', yaxis_title='Mean IGS Score', height=420,
)
fig.show()

## 6. Turnaround vs. Stuck — Community Typology

Among tracts with IGS < 45 in 2017, did they reach IGS ≥ 45 by 2025 (Turnaround) or remain below (Stuck)?

In [ ]:
if typology is None:
    print('Typology not available — run: python 03_analysis/run_all_analysis.py')
else:
    at_risk = typology[typology['igs_score_2017'] < IGS_VULN_THRESHOLD].copy()
    counts  = at_risk['typology'].value_counts()

    print('=== TURNAROUND vs. STUCK (at-risk tracts only) ===')
    for typ, n in counts.items():
        print(f'  {typ:<15}  {n:>7,}  ({n/len(at_risk)*100:.1f}%)')

    # Delta-specific
    if 'county_fips5' in at_risk.columns:
        delta_ar = at_risk[at_risk['county_fips5'].isin(DELTA_COUNTY_FIPS)]
        print(f'\nDelta at-risk tracts: {len(delta_ar):,}')
        delta_counts = delta_ar['typology'].value_counts()
        for typ, n in delta_counts.items():
            print(f'  {typ:<15}  {n:>5,}  ({n/len(delta_ar)*100:.0f}%)')

    # Turnaround vs Stuck: mean 2025 IGS
    if benchmarks is not None:
        print('\n=== TURNAROUND BENCHMARK (mean 2025 values) ===')
        turn_bm  = benchmarks[benchmarks['typology'] == 'Turnaround'].set_index('indicator')
        stuck_bm = benchmarks[benchmarks['typology'] == 'Stuck'].set_index('indicator')
        key_subs = [s for s in list(IGS_SUB_TO_PILLAR.keys()) if s in turn_bm.index and s in stuck_bm.index]
        if key_subs:
            print(f"  {'Sub-indicator':<45} {'Turnaround':>12} {'Stuck':>8} {'Δ':>8}")
            print('  ' + '-' * 76)
            for sub in key_subs:
                t = turn_bm.loc[sub, 'mean_2025']
                s = stuck_bm.loc[sub, 'mean_2025']
                print(f"  {sub:<45} {t:>12.1f} {s:>8.1f} {t-s:>+8.1f}")

## 7. SHAP Feature Importance — What Actually Drives Turnaround?

Loaded directly from `shap_feature_summary.parquet` and `shap_dimensions.parquet` (output of `ml_discovery.py`). Nothing is hardcoded.

In [ ]:
if shap_summary is None or shap_dims is None:
    print('SHAP outputs not available — run: python 03_analysis/ml_discovery.py')
else:
    print('=== TOP 20 FEATURES BY MEAN |SHAP| ===')
    print(f"  {'Rank':>4} {'Feature':<50} {'|SHAP|':>10} {'%Total':>7} {'Direction':>10}")
    print('  ' + '-' * 84)
    for _, row in shap_summary.head(20).iterrows():
        direction = 'positive' if row['mean_shap'] > 0 else 'negative'
        print(f"  {int(row['rank']):>4} {row['feature']:<50} "
              f"{row['mean_abs_shap']:>10.4f} {row['pct_total']:>6.1f}%  {direction:>10}")

    print('\n=== SHAP VULNERABILITY DIMENSIONS (data-driven clustering) ===')
    print(f"  {'Dimension':<40} {'Weight %':>9} {'# Features':>11} {'Top Feature'}")
    print('  ' + '-' * 85)
    for _, row in shap_dims.sort_values('weight_pct', ascending=False).iterrows():
        print(f"  {row['dimension_name']:<40} {row['weight_pct']:>8.1f}%  {row['n_features']:>10}   {row['top_feature']}")

    print('\n=== DIMENSION FEATURES (from shap_derived_weights.json) ===')
    for dim, info in shap_weights.items():
        print(f"\n  {dim}  (weight={info['weight']:.4f}, top={info['top_feature']})")
        for f in info['features']:
            print(f'    · {f}')

## 8. Compound Climate-Health Risk

In [ ]:
health_cols = {
    'CASTHMA_CrudePrev':    'Asthma',
    'CHD_CrudePrev':        'Heart Disease',
    'COPD_CrudePrev':       'COPD',
    'BPHIGH_CrudePrev':     'Hypertension',
    'DIABETES_CrudePrev':   'Diabetes',
    'DEPRESSION_CrudePrev': 'Depression',
    'OBESITY_CrudePrev':    'Obesity',
}
available_health = {k: v for k, v in health_cols.items() if k in df.columns}

if available_health:
    print(f"  {'Condition':<22} {'National Avg':>14} {'Delta Avg':>12} {'Ratio':>8}")
    print('  ' + '-' * 60)
    for col, label in available_health.items():
        nat_avg   = df[col].mean()
        delta_avg = delta_df[col].mean() if not delta_df.empty else float('nan')
        ratio     = delta_avg / nat_avg if nat_avg > 0 else float('nan')
        flag      = '  ← elevated >30%' if ratio > 1.3 else ''
        print(f"  {label:<22} {nat_avg:>13.1f}% {delta_avg:>11.1f}% {ratio:>7.2f}x{flag}")

    # Compound risk score: health burden percentile + FEMA risk percentile
    if 'RISK_SCORE' in df.columns:
        health_burden_score = df[list(available_health.keys())].mean(axis=1)
        df['health_burden_pct'] = health_burden_score.rank(pct=True) * 100
        df['climate_risk_pct']  = df['RISK_SCORE'].rank(pct=True) * 100
        df['compound_risk']     = (df['health_burden_pct'] + df['climate_risk_pct']) / 2

        r, p = pearsonr(
            df[['igs_score','compound_risk']].dropna()['igs_score'],
            df[['igs_score','compound_risk']].dropna()['compound_risk'],
        )
        print(f'\nCorrelation of compound_risk with IGS: r = {r:.3f} (p = {p:.2e})')
        print('Expected: negative (higher compound risk → lower IGS)')

## 9. Mississippi Delta Deep Dive

In [ ]:
if delta_df.empty:
    print('No Delta tracts found — check county_fips5 column in master_tract.parquet')
else:
    pillar_cols = [c for c in ['igs_economy', 'igs_place', 'igs_community'] if c in delta_df.columns]

    agg_spec = {
        'n_tracts':     ('GEOID', 'count'),
        'mean_igs':     ('igs_score', 'mean'),
        'pct_below_45': ('econ_vuln', 'mean'),
    }
    for col in pillar_cols:
        agg_spec[f'mean_{col}'] = (col, 'mean')

    delta_summary = (
        delta_df.groupby('county_fips5')
        .agg(**agg_spec)
        .reset_index()
        .assign(county_name=lambda x: x['county_fips5'].map(DELTA_COUNTY_NAMES))
        .sort_values('mean_igs')
    )
    delta_summary['pct_below_45'] = (delta_summary['pct_below_45'] * 100).round(0)

    display_cols = ['county_name', 'n_tracts', 'mean_igs', 'pct_below_45'] + \
                   [f'mean_{c}' for c in pillar_cols]
    print(delta_summary[display_cols].to_string(index=False))

    # IGS pillar radar for Bolivar vs Delta avg vs National avg
    if pillar_cols:
        bolivar_row  = delta_df[delta_df['county_fips5'] == '28011'][pillar_cols].mean()
        delta_avg    = delta_df[pillar_cols].mean()
        national_avg = df[pillar_cols].mean()

        labels = [c.replace('igs_', '').capitalize() for c in pillar_cols]
        cats   = labels + [labels[0]]  # close the radar loop

        fig = go.Figure()
        for vals, name, color in [
            (national_avg, 'National Average',   'grey'),
            (delta_avg,    'MS Delta Average',    '#d73027'),
            (bolivar_row,  'Bolivar County',      '#2166ac'),
        ]:
            v = list(vals) + [vals.iloc[0]]
            fig.add_trace(go.Scatterpolar(
                r=v, theta=cats, fill='toself', name=name,
                line=dict(color=color, width=2), opacity=0.7,
            ))
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
            title='IGS Pillar Scores: Bolivar County vs. Benchmarks',
            height=450,
        )
        fig.show()

## 10. Investment Simulator — IGS Sub-indicator Gap Analysis

For each Delta county, we compute the gap between its current sub-indicator values and the turnaround benchmark.  
If regression coefficients are available, we estimate the associational IGS gain from closing each gap.  
**These are correlational estimates from OLS, not causal projections.**

In [ ]:
if benchmarks is None:
    print('Benchmarks not available — run: python 03_analysis/run_all_analysis.py')
elif delta_df.empty:
    print('No Delta tracts found.')
else:
    turn_targets = (
        benchmarks[benchmarks['typology'] == 'Turnaround']
        .set_index('indicator')['mean_2025']
        .to_dict()
    )

    # Build coefficient lookup from regression output (if available)
    coef_lookup = {}
    if reg_coefs is not None and 'feature' in reg_coefs.columns and 'coeff_raw' in reg_coefs.columns:
        coef_lookup = reg_coefs.set_index('feature')['coeff_raw'].to_dict()
        print(f'Using {len(coef_lookup)} OLS regression coefficients loaded from igs_regression_coefficients.parquet')
    else:
        print('Regression coefficients not found — gap impact column will show N/A')

    for county_fips in sorted(DELTA_COUNTY_FIPS):
        county_name = DELTA_COUNTY_NAMES.get(county_fips, county_fips)
        county_data = delta_df[delta_df['county_fips5'] == county_fips]
        if county_data.empty:
            continue

        print(f'\n=== {county_name} County (FIPS {county_fips}) ===')
        print(f'  Mean IGS: {county_data["igs_score"].mean():.1f}')
        print(f"  {'Sub-indicator':<45} {'Current':>9} {'Target':>9} {'Gap':>7} {'OLS Gain':>10}")
        print('  ' + '-' * 84)

        for sub in IGS_SUB_TO_PILLAR:
            if sub not in county_data.columns or sub not in turn_targets:
                continue
            current = county_data[sub].mean()
            target  = turn_targets[sub]
            gap     = target - current
            if gap <= 0:
                continue  # already meets or exceeds benchmark

            coef = coef_lookup.get(sub, None)
            gain_str = f'{gap * coef:>+10.2f}' if coef is not None else '       N/A'
            print(f"  {sub:<45} {current:>9.1f} {target:>9.1f} {gap:>+7.1f} {gain_str}")

        print('  (OLS gain = gap × regression coefficient — associational, not causal)')

## 11. Key Numbers for Reporting

In [ ]:
print('=== KEY NUMBERS FOR REPORTING ===')
print()
print(f'Total tracts analyzed:              {len(df):,}')
print(f'Economically vulnerable (IGS<{IGS_VULN_THRESHOLD}):   '
      f'{int(df["econ_vuln"].sum()):,}  ({df["econ_vuln"].mean()*100:.1f}%)')
print(f'Triple-vulnerable tracts:           '
      f'{int(df["triple_vuln"].sum()):,}  ({df["triple_vuln"].mean()*100:.1f}%)')

if pop_col:
    print(f'People in triple-vuln tracts:       '
          f'{df[df["triple_vuln"]][pop_col].sum()/1e6:.1f}M')

print(f'National mean IGS:                  {df["igs_score"].mean():.1f} / 100')
if not delta_df.empty:
    print(f'MS Delta mean IGS:                  {delta_df["igs_score"].mean():.1f} / 100')
    print(f'Delta triple-vuln rate:             {delta_df["triple_vuln"].mean()*100:.0f}%')

if 'RISK_SCORE' in df.columns:
    valid = df[['igs_score', 'RISK_SCORE']].dropna()
    r, p  = pearsonr(valid['igs_score'], valid['RISK_SCORE'])
    print(f'IGS vs FEMA risk correlation:       r = {r:.3f}  (n={len(valid):,})')

if 'fqhc_count' in df.columns and 'county_fips5' in df.columns:
    delta_fqhc = delta_df.groupby('county_fips5')['fqhc_count'].first().sum()
    print(f'FQHCs in 9 Delta counties:          {delta_fqhc:.0f}')

if shap_dims is not None:
    print(f'\nSHAP dimensions (weights loaded from file):')
    for _, row in shap_dims.sort_values('weight_pct', ascending=False).iterrows():
        print(f'  {row["dimension_name"]:<40}  {row["weight_pct"]:.1f}%')